**Load Model and Run Inference**

In [ ]:
import torch
from PIL import Image
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# load image
image_path = r"..\data\formatted\license_plate_detection\test\images\lp_test_image105.jpg"
img = Image.open(image_path).convert("RGB")

# transform image to tensor
transform = T.Compose([
    T.ToTensor()
])

input_tensor = transform(img).unsqueeze(0)

# load model and replace classifier head
def get_object_detection_model(num_classes):
    # load base model
    model = fasterrcnn_mobilenet_v3_large_320_fpn(weights="DEFAULT")
    
    # replace model head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# instantiate model
model = get_object_detection_model(num_classes=2)

# load model weights
weights_path = r"..\models\lp_cnn.pt"
model.load_state_dict(torch.load(weights_path, map_location=torch.device('cpu')))


# set to eval
model.eval()

# select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#load model and input to device
model = model.to(device)
input_tensor = input_tensor.to(device)

# run input through model
with torch.no_grad():
    predictions = model(input_tensor)

# get boxes, labels, scores
boxes = predictions[0]['boxes'].cpu().numpy()
labels = predictions[0]['labels'].cpu().numpy()
scores = predictions[0]['scores'].cpu().numpy()

print("RAW SCORES:", predictions[0]['scores'].cpu().numpy())
print("NUMBER OF BOXES FOUND:", len(predictions[0]['boxes']))

**Plot Image With Bounding Box**

In [ ]:
# plot image with bounding box
fig, ax = plt.subplots(1, figsize=(10, 10))
ax.imshow(img)

# set threshold for bounding boxes
c_thresh = 0.2

for box, label, score in zip(boxes, labels, scores):
    if score > c_thresh:
        xmin, ymin, xmax, ymax = box
        width, height = xmax - xmin, ymax - ymin
        
        # add rectangle
        rect = patches.Rectangle((xmin, ymin), width, height, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        
        # add label
        plt.text(xmin, ymin - 5, f"Plate: {score:.2f}", color='white', bbox=dict(facecolor='red', alpha=0.5))

plt.axis('off')
plt.show()

In [ ]:
import pandas as pd

results_df = pd.read_csv(r"..\training_results\metrics.csv")

results_df = results_df.map(lambda x: x.item() if hasattr(x, 'item') else x)
